In [ ]:
import os

# Working directory must contain AlphaSimPy.py for imports
os.chdir(r"/Users/mtwatson/Library/CloudStorage/Box-Box/Projects/AI agent for breeding/Endpoint 2 agent")


# Atanda Plant Breeding Program - AlphaSimPy Notebook

This notebook converts the provided **BRAID** abstraction into a tutorial-style **AlphaSimPy** simulation.

It represents a **pedigree-style plant breeding program** that:
- starts with an elite × elite biparental cross,
- advances material from **F1 to F4** by selfing,
- evaluates and selects at **F5, F6, F7, F8, and F9**, and
- ends with **F10 release and seed increase**.

**Source abstraction**: BRAID YAML  
**Simulation framework**: AlphaSimPy  
**Trait modeled**: `overall_performance`


## Assumptions Used in the Translation

The BRAID abstraction intentionally leaves some biological and simulation details unspecified. To make the notebook executable, the following assumptions are used:

1. **Genome size**: the BRAID file lists `chromosomes: 0` and `n_qtl: 0`, so this notebook uses a minimal executable genome with **1 chromosome** and **500 QTL**.
2. **Founders**: the abstraction specifies **2 elite founders**, interpreted as two inbred parental lines.
3. **Crossing design**: one biparental elite × elite cross is created to form the F1.
4. **Advancement from F1 to F4**: implemented as repeated selfing with population expansion.
5. **Selection unit**: BRAID describes family-level phenotypic selection; here it is approximated with truncation selection on individual phenotypes within each stage.
6. **Multi-environment testing**: environments × replications are represented through the `reps` argument in `setPheno`, using the BRAID stage-specific trial intensity.
7. **Inbreeding output**: AlphaSimPy support can vary by build, so this notebook tracks a practical proxy using stage progression and reports genetic mean and variance directly.

These assumptions are stated explicitly so the notebook remains transparent and editable.


## Import Required Libraries

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from AlphaSimPy import (
    runMacs,
    SimParam,
    newPop,
    randCross,
    self,
    setPheno,
    selectInd,
    meanG,
    varG,
    meanP,
)

print('AlphaSimPy notebook for the Atanda Plant Breeding Program')
print('Libraries imported successfully.')


AlphaSimPy notebook for the Atanda Plant Breeding Program
Libraries imported successfully.


## Global Parameters

This section maps the BRAID program into explicit AlphaSimPy parameters.


In [6]:
# ---- Program horizon ----
horizon = 10
timestep_unit = 'generation'

# ---- Genome assumptions required for execution ----
n_chr = 1
n_qtl = 500
n_snp = 0
ploidy = 2

# ---- Trait assumptions from BRAID ----
trait_name = 'overall_performance'
trait_mean = 1.0
trait_var = 1.0
heritability = 0.3
error_variance = 1.0

# ---- Founder and crossing design ----
n_founders = 2
n_crosses = 1
parents_per_cross = 2
f1_progeny = 50

# ---- Stage sizes from BRAID ----
n_f5 = 4000
n_f6 = 400
n_f7 = 40
n_f8 = 40
n_f9 = 4
n_release = 1

# ---- Trial intensity from BRAID (environments x replications) ----
reps_f5 = 1 * 1
reps_f6 = 2 * 2
reps_f7 = 6 * 3
reps_f8 = 6 * 3
reps_f9 = 10 * 3

print('Parameters initialised')
print(f'Horizon: {horizon} {timestep_unit}s')
print(f'Genome: {n_chr} chromosome, {n_qtl} QTL')
print(f'Founders: {n_founders}')
print(f'Stage sizes F5/F6/F7/F8/F9: {n_f5}/{n_f6}/{n_f7}/{n_f8}/{n_f9}')


Parameters initialised
Horizon: 10 generations
Genome: 1 chromosome, 500 QTL
Founders: 2
Stage sizes F5/F6/F7/F8/F9: 4000/400/40/40/4


## Create Founders and Simulation Parameters

This follows the structure used in the AlphaSimPy line-breeding tutorials:
1. simulate founder haplotypes,
2. create `SimParam`,
3. define the additive trait, and
4. create the initial parent population.


In [7]:
founderPop = runMacs(
    nInd=n_founders,
    nChr=n_chr,
    segSites=n_qtl + n_snp,
    inbred=True,
    species='GENERIC'
)

SP = SimParam(founderPop)
SP.addTraitA(nQtlPerChr=n_qtl, mean=trait_mean, var=trait_var)
SP.setTrackPed(True)

parents = newPop(founderPop, simParam=SP)
parents = setPheno(parents, varE=error_variance, reps=reps_f9, simParam=SP)

print('Founders and parents created')
print(f'Number of parents: {parents.n_ind}')
print(f'Mean G (parents): {meanG(parents)[0]:.3f}')
print(f'Var G (parents): {varG(parents)[0]:.3f}')


Founders and parents created
Number of parents: 2
Mean G (parents): 1.000
Var G (parents): 1.000


## Helper Function for Stage Summaries

We collect stage-level summaries for the outputs requested in the BRAID abstraction.


In [8]:
records = []

def recordStage(stage_name, pop, generation):
    records.append({
        'generation': generation,
        'stage': stage_name,
        'nInd': pop.n_ind,
        'meanG': float(meanG(pop)[0]),
        'varG': float(varG(pop)[0]),
        'meanP': float(meanP(pop)[0]) if hasattr(pop, 'pheno') else np.nan,
    })
    print(f"Recorded {stage_name}: n={pop.n_ind}, meanG={meanG(pop)[0]:.3f}, varG={varG(pop)[0]:.3f}")


## Simulate the Atanda Breeding Pipeline

The pipeline below directly mirrors the BRAID workflow:

- `parents` → `f1` by one elite × elite cross
- `f1` → `f2_f4` by three generations of selfing
- `f2_f4` → `f5` by one more generation of selfing and expansion
- phenotypic evaluation and truncation selection through F5, F6, F7, F8, and F9
- final release of one line at F10


In [ ]:
# Generation 1: elite x elite cross
f1 = randCross(parents, nCrosses=n_crosses, nProgeny=f1_progeny, simParam=SP)
recordStage('F1', f1, 1)

# Generations 2-4: selfing advance through F2-F4 bulk phase
f2 = self(f1, nProgeny=20, simParam=SP)
f3 = self(f2, nProgeny=4, simParam=SP)
f2_f4 = self(f3, nProgeny=1, simParam=SP)
recordStage('F2-F4', f2_f4, 4)

# Generation 5: advance to F5 and expand to target size
f5_base = self(f2_f4, nProgeny=20, simParam=SP)
f5 = selectInd(f5_base, nInd=n_f5, use='gv', simParam=SP) if f5_base.n_ind >= n_f5 else self(f2_f4, nProgeny=int(np.ceil(n_f5 / max(1, f2_f4.n_ind))), simParam=SP)
if f5.n_ind > n_f5:
    f5 = selectInd(f5, nInd=n_f5, use='gv', simParam=SP)
f5 = setPheno(f5, varE=error_variance, reps=reps_f5, simParam=SP)
recordStage('F5', f5, 5)

# Select to F6
f6 = selectInd(f5, nInd=n_f6, use='pheno', simParam=SP)
f6 = setPheno(f6, varE=error_variance, reps=reps_f6, simParam=SP)
recordStage('F6', f6, 6)

# Select to F7
f7 = selectInd(f6, nInd=n_f7, use='pheno', simParam=SP)
f7 = setPheno(f7, varE=error_variance, reps=reps_f7, simParam=SP)
recordStage('F7', f7, 7)

# Select to F8
f8 = selectInd(f7, nInd=n_f8, use='pheno', simParam=SP)
f8 = setPheno(f8, varE=error_variance, reps=reps_f8, simParam=SP)
recordStage('F8', f8, 8)

# Select to F9
f9 = selectInd(f8, nInd=n_f9, use='pheno', simParam=SP)
f9 = setPheno(f9, varE=error_variance, reps=reps_f9, simParam=SP)
recordStage('F9', f9, 9)

# Release one line at F10
f10 = selectInd(f9, nInd=n_release, use='pheno', simParam=SP)
recordStage('F10 Release', f10, 10)

print('Pipeline complete.')


Recorded F1: n=50, meanG=1.107, varG=0.977
Recorded F2-F4: n=4000, meanG=1.128, varG=1.867


## Stage Summary Table

In [ ]:
summary = pd.DataFrame(records)
summary

## Plot Genetic Mean and Genetic Variance Across Stages

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(summary['generation'], summary['meanG'], marker='o')
axes[0].set_title('Genetic Mean by Stage')
axes[0].set_xlabel('Generation')
axes[0].set_ylabel('Mean G')
axes[0].grid(True, alpha=0.3)

axes[1].plot(summary['generation'], summary['varG'], marker='o', color='darkorange')
axes[1].set_title('Genetic Variance by Stage')
axes[1].set_xlabel('Generation')
axes[1].set_ylabel('Var G')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Final Notes

This notebook is a direct executable translation of the BRAID abstraction into an AlphaSimPy workflow.

Key preserved features from the BRAID program:
- elite × elite biparental start,
- F1 to F4 selfing advancement,
- stage-specific phenotypic evaluation intensity,
- truncation selection from F5 through release,
- tracking of genetic mean and genetic variance.

If desired, this notebook can be extended to include:
- explicit family-based selection,
- more realistic founder numbers,
- marker chips and genomic prediction,
- repeated yearly pipeline advancement.
